In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
from os import listdir

import matplotlib.pyplot as plt
from pendulibrary.continuation import adaptive_cont
from pendulibrary.common_targetters import single_fixed
from pendulibrary.plotters import compare_fast
from pendulibrary.integrate import integrate_state
from pendulibrary.common import hamiltonian
from tqdm.auto import tqdm

In [ ]:
fname = "DDsp-5a"

data = np.load(f"../database/{fname}.npz")
x0s_in = data["x0s"]
periods_in = data["periods"]
tangents_in = data["tangents"]
eigs_in = data["eigs"]
Lr, Mr = data["params"]

targ = single_fixed(0, x0s_in[0][0], 2, Lr, Mr, 1e-14)
func = targ.g_dg_stm

In [ ]:
Xm0 = targ.get_X(x0s_in[0], periods_in[0])
Xm1 = targ.get_X(x0s_in[1], periods_in[1])

g, dg, stm = func(Xm0)
svd = np.linalg.svd(dg)
svd.S

In [ ]:
# tangent = svd.Vh[-2]
# sprev = np.linalg.norm(Xm0 - Xm1)
# if np.dot(tangent, Xm1 - Xm0) / sprev < 0:
#     tangent *= -1
# sprev = np.linalg.norm(Xm0 - Xm1)
# print(f"Last dist is {sprev:.3e}")
tangent = svd.Vh[-1]

In [ ]:
cont_kwargs = dict(
    s0=1e-6, s_min=1e-7, S=25.0, tol=1e-9, max_iter=15, target_iter=7, rate=1.02
)
Xs, eig_vals, (DFs, tangents, stms) = adaptive_cont(
    Xm0, func, tangent, **cont_kwargs, exact_tangent=True
)

In [ ]:
fnames = [f.removesuffix(".npz") for f in listdir("../database") if "DDsp-P2" in f]
x0s_new = np.array([targ.get_x0(X) for X in Xs])
periods_new = np.array([targ.get_period(X) for X in Xs])
compare_fast(periods_new, hamiltonian(x0s_new.T, Lr, Mr), fnames)

In [ ]:
# np.savez(
#     "../database/DDlp-P4b-backward",
#     x0s=x0s_new,
#     periods=periods_new,
#     eigs=eig_vals,
#     tangents=tangents,
#     params=np.array([Lr, Mr]),
# )

In [ ]:
plt.close("all")
ts, xs, fs = integrate_state(x0s_new[-1], periods_new[-1], Lr, Mr, 1e-14)
y = -np.cos(xs[0]) - Lr * np.cos(xs[1])
x = np.sin(xs[0]) + Lr * np.sin(xs[1])

# plt.plot(xs.T)
plt.plot(x, y)
plt.axis("equal")
plt.show()

In [ ]:
from pendulibrary.plotters import make_gif

In [ ]:
make_gif(xs,ts,fs,Lr,"test")